In [1]:
"""
协同状态转移模型演示
"""

import sys
import os
from pathlib import Path

# 获取当前 notebook 所在的目录（Jupyter 中）
try:
    # 尝试获取 __file__（在 .py 文件中有效）
    current_dir = os.path.dirname(os.path.abspath(__file__))
except NameError:
    # 在 Jupyter notebook 中，使用当前工作目录
    current_dir = os.getcwd()
    
# 将当前目录添加到系统路径
sys.path.append(current_dir)

from models.synergistic_model import SynergisticStateTransitionModel
from models.base_model import EventType

In [2]:
def demo_synergistic_model():
    """演示协同状态转移模型"""
    print("=" * 80)
    print("协同状态转移模型演示")
    print("=" * 80)
    
    # 创建模型实例 - n=5, m=3 (联合干预更快)
    model = SynergisticStateTransitionModel(n=5, m=3)
    
    n_val = model.model_params['n']
    m_val = model.model_params['m']
    
    print(f"\n📊 模型信息:")
    print(f"   参数: n={n_val}, m={m_val} (联合干预比单一干预快{n_val - m_val}步)")
    print(f"   初始状态: (0,1,1) 不健康")
    print(f"   可用干预:")
    print(f"      - 单一干预: (0,0,1) 🔵")
    print(f"      - 联合干预: (0,1,1) 🟣")
    print(f"      - 无干预: (0,0,0) ⚪")
    
    print(f"\n📖 状态说明:")
    print(f"   (0,1,1) → 不健康状态")
    print(f"   (0,0,0) → 稳定健康状态")
    print(f"   (0.1,1) → 中间状态（不可观测）")
    
    print(f"\n📖 事件说明:")
    print(f"   E3a (单一干预转移): 连续 {n_val} 次单一干预 → 稳定健康")
    print(f"   E3b (协同转移): 连续 {m_val} 次联合干预 → 稳定健康（加速）")
    print(f"   E3c (无有效转移): 其他情况 → 中间状态 (0.1,1)")
    
    # 场景1: 单一干预达到 n 次
    print("\n" + "=" * 80)
    print(f"场景1: 单一干预 - 连续 {n_val} 次单一干预")
    print("=" * 80)
    
    interventions_scene1 = [(0, 0, 1)] * n_val
    
    result1 = model.simulate(interventions_scene1, return_details=True)
    _print_results(result1, model, title=f"连续{n_val}次单一干预 → 稳定健康")
    
    # 场景2: 联合干预达到 m 次（加速）
    print("\n" + "=" * 80)
    print(f"场景2: 联合干预 - 连续 {m_val} 次联合干预（加速）")
    print("=" * 80)
    
    interventions_scene2 = [(0, 1, 1)] * m_val
    
    result2 = model.simulate(interventions_scene2, return_details=True)
    _print_results(result2, model, title=f"连续{m_val}次联合干预 → 稳定健康（比单一干预快{n_val - m_val}步）")
    
    # 场景3: 未达到阈值
    print("\n" + "=" * 80)
    print(f"场景3: 未达到阈值 - 连续 {m_val-1} 次联合干预")
    print("=" * 80)
    
    interventions_scene3 = [(0, 1, 1)] * (m_val - 1)
    
    result3 = model.simulate(interventions_scene3, return_details=True)
    _print_results(result3, model, title=f"连续{m_val-1}次联合干预（未达到{m_val}次）→ 中间状态")
    
    # 场景4: 混合干预 - 单一干预被中断
    print("\n" + "=" * 80)
    print("场景4: 混合干预 - 单一干预序列被中断")
    print("=" * 80)
    
    interventions_scene4 = [
        (0, 0, 1),  # t=1: 单一干预
        (0, 0, 1),  # t=2: 单一干预
        (0, 0, 1),  # t=3: 单一干预
        (0, 1, 1),  # t=4: 联合干预（中断单一干预序列）
        (0, 1, 1),  # t=5: 联合干预
        (0, 1, 1),  # t=6: 联合干预（达到 m=3，触发协同转移）
    ]
    
    result4 = model.simulate(interventions_scene4, return_details=True)
    _print_results(result4, model, title="单一干预序列被联合干预中断，联合干预达到阈值 → 稳定健康")
    
    # 场景5: 对比单一干预和联合干预
    print("\n" + "=" * 80)
    print("场景5: 对比演示 - 相同步数下不同干预的效果")
    print("=" * 80)
    
    _compare_interventions(model)


def _compare_interventions(model):
    """对比单一干预和联合干预的效果"""
    n_val = model.model_params['n']
    m_val = model.model_params['m']
    steps = max(n_val, m_val)
    
    print(f"\n对比 {steps} 步内不同干预序列的效果:")
    print("-" * 60)
    
    # 单一干预序列
    single_interventions = [(0, 0, 1)] * steps
    result_single = model.simulate(single_interventions)
    
    # 联合干预序列
    joint_interventions = [(0, 1, 1)] * steps
    model.reset_internal_state()
    result_joint = model.simulate(joint_interventions)
    
    # 无干预序列
    none_interventions = [(0, 0, 0)] * steps
    model.reset_internal_state()
    result_none = model.simulate(none_interventions)
    
    print(f"\n{'干预类型':<15} {'最终状态':<20} {'是否健康':<10}")
    print("-" * 60)
    
    final_single = result_single["states"][-1]
    final_joint = result_joint["states"][-1]
    final_none = result_none["states"][-1]
    
    single_healthy = "是" if final_single == model.HEALTHY else "否"
    joint_healthy = "是" if final_joint == model.HEALTHY else "否"
    none_healthy = "是" if final_none == model.HEALTHY else "否"
    
    print(f"{'单一干预':<15} {str(final_single):<20} {single_healthy:<10}")
    print(f"{'联合干预':<15} {str(final_joint):<20} {joint_healthy:<10}")
    print(f"{'无干预':<15} {str(final_none):<20} {none_healthy:<10}")
    
    print(f"\n💡 结论:")
    print(f"   - 联合干预只需要 {m_val} 步即可达到健康")
    print(f"   - 单一干预需要 {n_val} 步才能达到健康")
    print(f"   - 联合干预比单一干预快 {n_val - m_val} 步")


def demo_threshold_comparison():
    """阈值对比演示"""
    print("\n" + "=" * 80)
    print("阈值对比：不同参数设置的影响")
    print("=" * 80)
    
    param_sets = [
        {"n": 4, "m": 2, "name": "n=4, m=2 (快2步)"},
        {"n": 6, "m": 3, "name": "n=6, m=3 (快3步)"},
        {"n": 8, "m": 4, "name": "n=8, m=4 (快4步)"},
    ]
    
    # 测试序列：3次联合干预
    test_interventions = [(0, 1, 1)] * 3
    
    for params in param_sets:
        model = SynergisticStateTransitionModel(n=params["n"], m=params["m"])
        result = model.simulate(test_interventions)
        final_state = result["states"][-1]
        
        # 检查是否触发完全转移
        triggered = final_state == model.HEALTHY
        
        print(f"\n{params['name']}:")
        print(f"   3次联合干预后最终状态: {final_state}")
        print(f"   是否达到健康: {'是' if triggered else '否'}")
        
        if params["m"] <= 3:
            print(f"   💡 m={params['m']} ≤ 3: 3次联合干预足以触发协同转移")
        else:
            print(f"   💡 m={params['m']} > 3: 3次联合干预不足以触发协同转移，需要{params['m']}次")


def _print_results(result: dict, model, title: str = ""):
    """打印模拟结果"""
    if title:
        print(f"\n{title}")
    
    print(f"\n干预序列:")
    for t, inter in enumerate(result["interventions"], 1):
        if inter == model.SINGLE_INTERVENTION:
            print(f"   t={t:2d}: {inter} 🔵 单一干预")
        elif inter == model.JOINT_INTERVENTION:
            print(f"   t={t:2d}: {inter} 🟣 联合干预")
        else:
            print(f"   t={t:2d}: {inter} ⚪ 无干预")
    
    print(f"\n状态演变:")
    print("-" * 80)
    print(f"{'步数':<6} {'可观测状态':<20} {'状态说明':<15} {'事件':<40}")
    print("-" * 80)
    
    for i in range(len(result["states"])):
        state = result["states"][i]
        
        if state == model.HEALTHY:
            state_name = "健康"
        elif state == model.UNHEALTHY:
            state_name = "不健康"
        else:
            state_name = "中间状态"
        
        if i == 0:
            event_str = "-"
        else:
            event = result["events"][i-1]
            if event == EventType.COMPLETE_TRANSITION:
                event_str = "E3a (单一干预转移) ⭐"
            elif event == EventType.SYNERGISTIC_TRANSITION:
                event_str = "E3b (协同转移) ⭐"
            else:
                event_str = "E3c (无有效转移)"
        
        display_state = f"{state} ({state_name})"
        print(f"{i:<6} {display_state:<20} {state_name:<15} {event_str:<40}")
    
    # 显示内部状态
    internal = model.get_internal_state()
    if internal["is_in_intermediate"]:
        print(f"\n⚠️  当前处于中间状态 (0.1,1)")

In [3]:
if __name__ == "__main__":
    demo_synergistic_model()
    demo_threshold_comparison()

协同状态转移模型演示

📊 模型信息:
   参数: n=5, m=3 (联合干预比单一干预快2步)
   初始状态: (0,1,1) 不健康
   可用干预:
      - 单一干预: (0,0,1) 🔵
      - 联合干预: (0,1,1) 🟣
      - 无干预: (0,0,0) ⚪

📖 状态说明:
   (0,1,1) → 不健康状态
   (0,0,0) → 稳定健康状态
   (0.1,1) → 中间状态（不可观测）

📖 事件说明:
   E3a (单一干预转移): 连续 5 次单一干预 → 稳定健康
   E3b (协同转移): 连续 3 次联合干预 → 稳定健康（加速）
   E3c (无有效转移): 其他情况 → 中间状态 (0.1,1)

场景1: 单一干预 - 连续 5 次单一干预

连续5次单一干预 → 稳定健康

干预序列:
   t= 1: (0, 0, 1) 🔵 单一干预
   t= 2: (0, 0, 1) 🔵 单一干预
   t= 3: (0, 0, 1) 🔵 单一干预
   t= 4: (0, 0, 1) 🔵 单一干预
   t= 5: (0, 0, 1) 🔵 单一干预

状态演变:
--------------------------------------------------------------------------------
步数     可观测状态                状态说明            事件                                      
--------------------------------------------------------------------------------
0      (0, 1, 1) (不健康)      不健康             -                                       
1      (0.1, 1) (中间状态)      中间状态            E3c (无有效转移)                             
2      (0.1, 1) (中间状态)      中间状态            E3c (无有效转移)  